In [ ]:
#this notebook uses the dataframes created in the previous notebook
!pip install mplfinance
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from scipy import stats
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import seaborn as sns
import mplfinance as mpf
import warnings
warnings.filterwarnings('ignore')

In [ ]:
cd = pd.read_csv('/content/drive/MyDrive/cd.csv') # cleaned database
df = pd.read_csv('/content/drive/MyDrive/df.csv') # feature engineered dataframe

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])

In [ ]:
cd.head(15)

,date,time,open,high,low,close,volume,open_interest
0,2017-01-02,09:15:00,100.00,100.28,100.00,100.02,7000.0,100000.0
1,2017-01-02,09:16:00,100.02,100.07,100.02,100.06,3509.0,101212.0
2,2017-01-02,09:17:00,100.05,100.06,99.70,99.73,5105.0,101212.0
3,2017-01-02,09:18:00,99.71,99.79,99.63,99.79,4959.0,101212.0
4,2017-01-02,09:19:00,99.78,99.82,99.69,99.70,2398.0,102480.0
5,2017-01-02,09:20:00,99.69,99.71,99.55,99.55,5011.0,102480.0
6,2017-01-02,09:21:00,99.57,99.73,99.54,99.73,3918.0,102480.0
7,2017-01-02,09:22:00,99.73,99.73,99.58,99.67,2784.0,103144.0
8,2017-01-02,09:23:00,99.67,99.70,99.58,99.63,1854.0,103144.0
9,2017-01-02,09:24:00,99.63,99.64,99.57,99.63,1257.0,103144.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672260 entries, 0 to 672259
Data columns (total 32 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   datetime             672260 non-null  datetime64[ns]
 1   day                  672260 non-null  int64         
 2   time                 672260 non-null  int64         
 3   oc                   672260 non-null  float64       
 4   hl                   672260 non-null  float64       
 5   vol                  672260 non-null  float64       
 6   opin                 672260 non-null  float64       
 7   ema                  672260 non-null  float64       
 8   opnorm               672260 non-null  float64       
 9   hinorm               672260 non-null  float64       
 10  lonorm               672260 non-null  float64       
 11  clnorm               672260 non-null  float64       
 12  ema_1st_grad         672260 non-null  float64       
 13  ema_2nd_grad  

In [ ]:
cd.tail(15)

,date,time,open,high,low,close,volume,open_interest
672245,2024-05-24,15:15:00,269.43,269.51,269.28,269.39,3688.0,111354.0
672246,2024-05-24,15:16:00,269.44,269.51,269.41,269.44,1243.0,111900.0
672247,2024-05-24,15:17:00,269.42,269.48,269.37,269.38,544.0,111900.0
672248,2024-05-24,15:18:00,269.38,269.43,269.30,269.35,934.0,111900.0
672249,2024-05-24,15:19:00,269.36,269.46,269.35,269.42,866.0,111995.0
672250,2024-05-24,15:20:00,269.42,269.47,269.40,269.46,835.0,111995.0
672251,2024-05-24,15:21:00,269.46,269.47,269.41,269.45,792.0,111995.0
672252,2024-05-24,15:22:00,269.45,269.45,269.36,269.41,590.0,111655.0
672253,2024-05-24,15:23:00,269.39,269.40,269.36,269.40,827.0,111655.0
672254,2024-05-24,15:24:00,269.40,269.43,269.36,269.37,2121.0,111655.0


In [ ]:
df.tail(15)

,datetime,day,time,oc,hl,vol,opin,ema,opnorm,hinorm,...,50sma-200sma,ultimate_oscillator,disparity_index,obv,a/d,rsi,adx,vol_freq,hi_freq,lo_freq
672245,2024-05-24 15:15:00,4,15,-0.04,0.23,3688.0,111354.0,269.422920,0.007080,0.087080,...,0.41035,51.660095,-0.024494,8660800.0,3.915049e+06,172.093023,25.116929,1.0,1.0,1.0
672246,2024-05-24 15:16:00,4,15,0.00,0.10,1243.0,111900.0,269.426025,0.013975,0.083975,...,0.40955,50.307451,-0.018183,8662043.0,3.914552e+06,193.939394,21.263270,1.0,2.0,2.0
672247,2024-05-24 15:17:00,4,15,-0.04,0.11,544.0,111900.0,269.417657,0.002343,0.062343,...,0.40630,44.053136,-0.043414,8661499.0,3.914106e+06,562.500000,19.025469,1.0,1.0,1.0
672248,2024-05-24 15:18:00,4,15,-0.03,0.13,934.0,111900.0,269.405356,-0.025356,0.024644,...,0.40405,42.985809,-0.051579,8660565.0,3.913891e+06,-400.000000,17.192136,1.0,1.0,1.0
672249,2024-05-24 15:19:00,4,15,0.06,0.11,866.0,111995.0,269.408018,-0.048018,0.051982,...,0.40190,44.000508,-0.022265,8661431.0,3.914127e+06,-344.444444,15.328118,1.0,1.0,1.0
672250,2024-05-24 15:20:00,4,15,0.04,0.07,835.0,111995.0,269.417470,0.002530,0.052530,...,0.40215,46.476924,0.000000,8662266.0,3.914724e+06,-100.000000,18.898968,1.0,1.0,1.0
672251,2024-05-24 15:21:00,4,15,-0.01,0.06,792.0,111995.0,269.423384,0.036616,0.046616,...,0.40280,48.914387,0.000742,8661474.0,3.914988e+06,-166.666667,22.591276,1.0,2.0,2.0
672252,2024-05-24 15:22:00,4,15,-0.04,0.09,590.0,111655.0,269.420951,0.029049,0.029049,...,0.40100,48.402514,-0.008908,8660884.0,3.915053e+06,-142.857143,26.136730,1.0,1.0,1.0
672253,2024-05-24 15:23:00,4,15,0.01,0.04,827.0,111655.0,269.417141,-0.027141,-0.017141,...,0.40075,50.698621,-0.005568,8660057.0,3.915880e+06,-84.210526,34.249938,1.0,1.0,2.0
672254,2024-05-24 15:24:00,4,15,-0.03,0.07,2121.0,111655.0,269.408570,-0.008570,0.021430,...,0.40075,50.929666,-0.013734,8657936.0,3.914365e+06,-200.000000,40.627716,1.0,2.0,3.0


In [ ]:
traincd, testcd = train_test_split(cd, test_size=0.2, train_size=0.8)
traindf, testdf = train_test_split(df, test_size=0.2, train_size=0.8)

In [ ]:
#clustering df
cdf = pd.DataFrame()
epsilon = 1e-10
bodylength = df['oc'] / cd['open']
cdf['candlelength'] = df['hl'] / cd['low']
cdf['bodypct'] = bodylength / (cdf['candlelength'] + epsilon)
lowerwick = (cd[['open', 'close']].min(axis=1) - cd['low']) / cd['open']
upperwick = (cd['high'] - cd[['open', 'close']].max(axis=1)) / cd['open']
cdf['uwpct'] = upperwick / (cdf['candlelength'] + epsilon)
cdf['lwpct'] = abs(lowerwick) / (cdf['candlelength'] + epsilon)
cdf['vol_1st_grad'] = df['vol_1st_grad']
#ccdf = cdf.dropna()
cdf.head(15)

,candlelength,bodypct,uwpct,lwpct,vol_1st_grad
0,0.002800,0.071429,0.928571,0.000000,7000.0
1,0.000500,0.800000,0.200000,0.000000,-3491.0
2,0.003611,-0.885779,0.027681,0.083042,1596.0
3,0.001606,0.499599,0.000000,0.499599,-146.0
4,0.001304,-0.614830,0.307415,0.076854,-2561.0
5,0.001607,-0.873771,0.124824,0.000000,2613.0
6,0.001909,0.841851,0.000000,0.157847,-1093.0
7,0.001506,-0.399398,0.000000,0.599098,-1134.0
8,0.001205,-0.333032,0.249774,0.416290,-930.0
9,0.000703,0.000000,0.142771,0.856627,-597.0


In [ ]:
cdf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 672260 entries, 0 to 672259
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   candlelength  672260 non-null  float64
 1   bodypct       672260 non-null  float64
 2   uwpct         672260 non-null  float64
 3   lwpct         672260 non-null  float64
 4   vol_1st_grad  672260 non-null  float64
dtypes: float64(5)
memory usage: 25.6 MB


In [ ]:
kmeans = KMeans(n_clusters=6, random_state=101)
kmeans.fit(cdf)
cdf['labels'] = kmeans.labels_

#this plot gives no information
'''u_labels = np.unique(labels)
pcdf = PCA(2).fit_transform(cdf)
for i in u_labels:
    plt.scatter(pcdf[labels == i, 0], pcdf[labels == i, 1], label = i)
plt.legend()
plt.show()'''

#visualisation is very important for clarity
cdf['index'] = cdf.index
'''cols = ['open', 'high', 'low', 'close']
temp_df = cd[cols]
temp_df['datetime'] = df['datetime']'''
temp_df = pd.DataFrame({'datetime': df['datetime'], 'open': df['opnorm'], 'high': df['hinorm'], 'low': df['lonorm'], 'close': df['clnorm']})
#non normalised prices didn't show any trends
temp_df['index'] = temp_df.index
vcdf = pd.merge(cdf, temp_df, on='index')
cols_ohlc = ['datetime', 'open', 'high', 'low', 'close', 'labels', 'vol_1st_grad']
temp_df = vcdf[cols_ohlc]

In [ ]:
temp_df0 = temp_df[temp_df['labels']==0]

fig0 = go.Figure(data=[go.Candlestick(x=temp_df0['datetime'],
                                     open=temp_df0['open'],
                                     high=temp_df0['high'],
                                     low=temp_df0['low'],
                                     close=temp_df0['close'])])
fig0.show()

In [ ]:
temp_df1 = temp_df[temp_df['labels']==1]

fig1 = go.Figure(data=[go.Candlestick(x=temp_df1['datetime'],
                                     open=temp_df1['open'],
                                     high=temp_df1['high'],
                                     low=temp_df1['low'],
                                     close=temp_df1['close'])])
fig1.show()

In [ ]:
temp_df2 = temp_df[temp_df['labels']==2]

fig2 = go.Figure(data=[go.Candlestick(x=temp_df2['datetime'],
                                     open=temp_df2['open'],
                                     high=temp_df2['high'],
                                     low=temp_df2['low'],
                                     close=temp_df2['close'])])
fig2.show()

In [ ]:
temp_df3 = temp_df[temp_df['labels']==3]

fig3 = go.Figure(data=[go.Candlestick(x=temp_df3['datetime'],
                                     open=temp_df3['open'],
                                     high=temp_df3['high'],
                                     low=temp_df3['low'],
                                     close=temp_df3['close'])])
fig3.show()

In [ ]:
temp_df4 = temp_df[temp_df['labels']==4]

fig4 = go.Figure(data=[go.Candlestick(x=temp_df4['datetime'],
                                     open=temp_df4['open'],
                                     high=temp_df4['high'],
                                     low=temp_df4['low'],
                                     close=temp_df4['close'])])
fig4.show()

In [ ]:
temp_df5 = temp_df[temp_df['labels']==5]

fig5 = go.Figure(data=[go.Candlestick(x=temp_df5['datetime'],
                                     open=temp_df5['open'],
                                     high=temp_df5['high'],
                                     low=temp_df5['low'],
                                     close=temp_df5['close'])])
fig5.show()